# Обработка таблицы 2100 отчетной статистической формы № 33 "Сведения о больных туберкулезом" за  3 года 2019-2021

Дата: 08.04.2026

Разработано в рамках 1 Этапа проекта **"Разработка системы визуализации медицинской статистики"**

**Цель:** 

1. Извлечение данных из исходных csv-файлов.
2. Приведение таблицы из "широкого", табличного, вида в "длинный".

**Шаблон результата:**
- "Формы туберкулеза"
- "Уточнение"
- "Еще уточнение"
- "Код по МКБ-Х пересмотра"
- "Год",
- "Значение",
- "Строка",
- "Графа"

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Настройки отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

In [3]:
path = 'd:/datasets/tuberculosis/'

In [4]:
def load_data_from_list(file_name, folder=path):
    """
    Функция загружает файл из списка file_name из папки folder
    """
    full_path = f"{folder}{file_name}"
    try:
        df = pd.read_csv(full_path, encoding='utf-8', encoding_errors='ignore')
        display(f"Файл загружен успешно: {file_name} | Строк: {df.shape[0]}")
        return df
    except Exception as e:
        print(f"Ошибка при загрузке: {file_name}: {e}")
        return None

In [5]:
def modify_2100 (df2100, year, folder_out):
    """
    Функция загружает предобработанный датасет таблицы 2100 из которого предварительно удалены пустые столбцы и шапка
    и преобразует его к длинному формату, добавляя номер строки и графу и частично заменяя названия в соответствии с выданным шаблоном
    """
   
    # Переименовать столбцы
    df2100.rename(columns={df2100.columns[0]: 'Формы туберкулеза',
                           df2100.columns[1]: 'Строка',
                           df2100.columns[2]: 'Код по МКБ-Х пересмотра',
                           df2100.columns[3]: 'g4',
                           df2100.columns[4]: 'g5',
                           df2100.columns[5]: 'g6',
                           df2100.columns[6]: 'g7',
                           df2100.columns[7]: 'g8',
                           df2100.columns[8]: 'g9',}, inplace=True)
    
    # Заменяем NaN на 0 во всей числовой части таблицы
    cols = ['g4', 'g5', 'g6', 'g7', 'g8', 'g9']
    df2100[cols] = df2100[cols].fillna(0)
    
    # Заменяем NaN на пробел в текстовом столбце
    cols = ['Код по МКБ-Х пересмотра']
    df2100[cols] = df2100[cols].fillna('')
    
    # Преобразовать значения граф в целое число
    cols = ['Строка','g4', 'g5', 'g6', 'g7', 'g8', 'g9']
    df2100[cols] = df2100[cols].astype(int)
    
    # Разворачиваем таблицу
    df_long = df2100.melt(
        id_vars=['Формы туберкулеза', 'Код по МКБ-Х пересмотра', 'Строка'], # Эти колонки оставляем как есть
        value_vars=['g4', 'g5', 'g6', 'g7', 'g8', 'g9'],                    # Эти колонки "схлопываем" в одну
        var_name='Показатель',                                              # Название новой колонки с именами g4, g5...
        value_name='Значение'                                               # Название новой колонки с числами
    )
    
    # Добавляем колонку "Уточнение"
    # insert(индекс_позиции, 'название', значения)
    df_long.insert(1, 'Уточнение', '')
    
    # Добавляем колонку "Еще уточнение"
    df_long.insert(2, 'Еще уточнение', '')
    
    # Добавляем колонку "Графа"
    df_long.insert(7, 'Графа', 0)
    
    # Указываем нужный порядок столбцов
    new_order = ['Формы туберкулеза', 'Уточнение', 'Еще уточнение', 'Код по МКБ-Х пересмотра', 'Показатель', 'Значение', 'Строка', 'Графа']
    df_long = df_long[new_order]
    
    # Заполоняем номер графы на основании столбца "Показатель"
    df_long['Графа'] = df_long['Показатель'].str[1:].astype(int)
    # Для граф 4, 5 и 6 - заполняем 'Уточнение' 
    df_long.loc[df_long['Показатель'].isin(['g4', 'g5', 'g6']), 'Уточнение'] = 'Взято на учет в отчетном году больных с первые в жизни установленным диагнозом'
    # Для граф 7, 8 и 9 - заполняем 'Уточнение' 
    df_long.loc[df_long['Показатель'].isin(['g7', 'g8', 'g9']), 'Уточнение'] = 'Контингенты больных на конец отчетного года'
    # Для графы 4 и 7 - Уточнение = 'всего'
    df_long.loc[df_long['Показатель'].isin(['g4', 'g7']), 'Еще уточнение'] = 'всего'
    # Для графы 5 и 8 - Уточнение = 'из них детей от 0 до 14 лет'
    df_long.loc[df_long['Показатель'].isin(['g5', 'g8']), 'Еще уточнение'] = 'из них детей от 0 до 14 лет'
    # Для графы 6 и 9 - Уточнение = 'из них подростков 15-17 лет'
    df_long.loc[df_long['Показатель'].isin(['g6', 'g9']), 'Еще уточнение'] = 'из них подростков 15-17 лет'
    
    # Переименовать столбцы, добавляем год
    df_long = df_long.rename(columns={'Показатель': 'Год'})
    df_long['Год'] = year
        
    # Преобразуем наименования с соответствии с шаблоном  - костыли
    df_long.loc[df_long['Строка'] == 2, 'Формы туберкулеза'] = 'Туберкулез органов дыхания - в том числе туберкулез легких'
    df_long.loc[df_long['Строка'] == 3, 'Формы туберкулеза'] = 'Туберкулез органов дыхания - из него: фиброзно-кавернозный'
    df_long.loc[df_long['Строка'] == 9, 'Формы туберкулеза'] = 'Имеют инвалидность в связи с туберкулезом в том числе первой группы'
    df_long.loc[df_long['Строка'] == 10, 'Формы туберкулеза'] = 'Имеют инвалидность в связи с туберкулезом в том числе второй группы'  
    df_long.loc[df_long['Строка'] == 12, 'Формы туберкулеза'] = 'Обследовано на АТ к ВИЧ из них с положительным результатом методом иммунного блотинга'  

    # Удаляем двойные пробелы в колонке 'Формы туберкулеза'
    df_long['Формы туберкулеза'] = df_long['Формы туберкулеза'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    ## Выгрузка
    df_long.to_csv(folder_out+'2100_'+str(year)+'.csv', index=False)
    return None

## 2019 год Таблица 2100 Формы 33

In [6]:
df2100 = load_data_from_list('2100.csv','d:/datasets/tuberculosis/2019/')

'Файл загружен успешно: 2100.csv | Строк: 18'

In [7]:
df2100.head(7)

,"1. Контингенты больных активным туберкулезом, состоящих под наблюдением данного лечебно-профилактического учреждения",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,(2100),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Код по ОКЕИ: человек - 792,NaN
1,Формы туберкулеза,№ стро-ки,Код по МКБ-Х пересмотра,Взято на учет в отчетном году больных с первые...,NaN,NaN,NaN,NaN,NaN,Контингенты больных на конец отчетного года,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,всего,NaN,из них:,NaN,NaN,NaN,всего,NaN,в том числе:,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN,NaN,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN
4,1,2,3,4,NaN,5,NaN,6,NaN,7,NaN,8,NaN,9,NaN
5,Туберкулез органов дыхания - всего,1,А15; А16; А19 часть,1557,NaN,36,NaN,22,NaN,3944,NaN,45,NaN,23,NaN
6,в том числе туберкулез легких,2,А15.0-А15.3; А15.7 часть; А16.0-А16.2; А16.7 ч...,1484,NaN,18,NaN,19,NaN,3853,NaN,28,NaN,21,NaN


In [8]:
# Удяляем пустые столбцы из таблицы 
df2100.dropna(axis=1, how='all', inplace=True)

In [9]:
# Удаление нескольких строк по списку индексов - удаляем шапку, номер графы тоже удаляем
df2100 = df2100.drop([0, 1, 2, 3, 4])

In [10]:
modify_2100(df2100, 2019, 'd:/datasets/tuberculosis/Свод/')

## 2020 год Таблица 2100 Формы 33

In [11]:
df2100 = load_data_from_list('2100.csv','d:/datasets/tuberculosis/2020/')

'Файл загружен успешно: 2100.csv | Строк: 18'

In [12]:
df2100.head(7)

,"1. Контингенты больных активным туберкулезом, состоящих под наблюдением данного лечебно-профилактического учреждения",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,(2100),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Код по ОКЕИ: человек - 792,NaN
1,Формы туберкулеза,№ стро-ки,Код по МКБ-Х пересмотра,Взято на учет в отчетном году больных с первые...,NaN,NaN,NaN,NaN,NaN,Контингенты больных на конец отчетного года,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,всего,NaN,из них:,NaN,NaN,NaN,всего,NaN,в том числе:,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN,NaN,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN
4,1,2,3,4,NaN,5,NaN,6,NaN,7,NaN,8,NaN,9,NaN
5,Туберкулез органов дыхания - всего,1,А15; А16; А19 часть,1230,NaN,25,NaN,20,0.0,3059,0.0,18,0.0,18,0.0
6,в том числе туберкулез легких,2,А15.0-А15.3; А15.7 часть; А16.0-А16.2; А16.7 ч...,1180,NaN,14,NaN,17,0.0,3000,0.0,12,0.0,15,0.0


In [13]:
# Удаление нескольких строк по списку индексов - удаляем шапку, номер графы тоже удаляем
df2100 = df2100.drop([0, 1, 2, 3, 4])

In [14]:
# Удяляем пустые столбцы из таблицы 
df2100.dropna(axis=1, how='all', inplace=True)

In [15]:
# Удаляем ненужные столбцы по номеру 
df2100 = df2100.drop(df2100.columns[[4, 6, 8, 10, 12, 14]], axis=1)

In [16]:
modify_2100(df2100, 2020, 'd:/datasets/tuberculosis/Свод/')

## 2021 год Таблица 2100 Формы 33

In [17]:
df2100 = load_data_from_list('2100.csv','d:/datasets/tuberculosis/2021/')

'Файл загружен успешно: 2100.csv | Строк: 18'

In [18]:
df2100.head(7)

,"1. Контингенты больных активным туберкулезом, состоящих под наблюдением данного лечебно-профилактического учреждения",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,(2100),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Код по ОКЕИ: человек - 792,NaN
1,Формы туберкулеза,№ стро-ки,Код по МКБ-Х пересмотра,Взято на учет в отчетном году больных с первые...,NaN,NaN,NaN,NaN,NaN,Контингенты больных на конец отчетного года,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,всего,NaN,из них:,NaN,NaN,NaN,всего,NaN,в том числе:,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN,NaN,NaN,детей 0-14 лет,NaN,подростков 15-17 лет,NaN
4,1,2,3,4,NaN,5,NaN,6,NaN,7,NaN,8,NaN,9,NaN
5,Туберкулез органов дыхания - всего,1,А15; А16; А19 часть,1122,NaN,27,NaN,12,0.0,2528,0.0,18,0.0,11,0.0
6,в том числе туберкулез легких,2,А15.0-А15.3; А15.7 часть; А16.0-А16.2; А16.7 ч...,1075,NaN,10,NaN,9,0.0,2487,0.0,8,0.0,9,0.0


In [19]:
# Удаление нескольких строк по списку индексов - удаляем шапку, номер графы тоже удаляем
df2100 = df2100.drop([0, 1, 2, 3, 4])

In [20]:
# Удяляем пустые столбцы из таблицы 
df2100.dropna(axis=1, how='all', inplace=True)

In [21]:
# Удаляем ненужные столбцы по номеру 
df2100 = df2100.drop(df2100.columns[[4, 6, 8, 10, 12, 14]], axis=1)

In [22]:
modify_2100(df2100, 2021, 'd:/datasets/tuberculosis/Свод/')

## Объединение таблиц

In [23]:
path = 'd:/datasets/tuberculosis/Свод/'

In [24]:
# Cписок таблиц
list_files = ['2100_2019.csv', '2100_2020.csv', '2100_2021.csv' ]

In [25]:
# Создаем пустрой датафрейм
df_result = pd.DataFrame()

In [26]:
for i in range(0, len(list_files)):
    # Читаем файл
    df2100 = load_data_from_list(list_files[i], path)
    # Объединяем 
    df_result = pd.concat([df_result, df2100], ignore_index=True)

'Файл загружен успешно: 2100_2019.csv | Строк: 78'

'Файл загружен успешно: 2100_2020.csv | Строк: 78'

'Файл загружен успешно: 2100_2021.csv | Строк: 78'

In [27]:
## Выгружаем
df_result.to_csv('d:/datasets/tuberculosis/Свод/'+'2100_2019-2021.csv', index=False)

**Результат**: предобработанный файл **2100_2019-2021.csv** с таблицей 2100 отчетной формы № 33 за 2019-2021 годы в "длинном" формате, необходимый для дальнейшей  обработки.